In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df_maths = pd.read_csv(r"..\data\student-mat.csv", delimiter=";")
df_por = pd.read_csv(r"..\data\student-por.csv", delimiter=";")

# adding one more column for subject
df_maths['subject'] = 'Maths'
df_por['subject']= 'Portuguese'

In [3]:
df_maths.columns == df_por.columns
# all columns are same

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True])

In [4]:
df = pd.concat([df_maths, df_por], axis=0).reset_index(drop=True)

### broblem with concatinating ?

few students ahve taken both maths and potugeues, so if we do simple k-fold cross validation, there will be problem of data leakage as same student can fall in both train and validation set.

### solution
- first lets create a unique it of each student
- use group k-fold cross validation

### GroupKFold CV
ensure that all records belonging to a specific student_id stay together in either the train or test set.

In [5]:
# Columns that identify a unique student
id_cols = ["school", "sex", "age", "address", "famsize", "Pstatus", 
           "Medu", "Fedu", "Mjob", "Fjob", "reason", "nursery", "internet"]

# Creating a 'student_id' column by combining these features
# this ensures that 'Student A' gets the same ID in both subjects
df['student_id'] = df[id_cols].astype(str).agg('-'.join, axis=1)

print(f"total rows: {len(df)}")
print(f"total unique students: {df['student_id'].nunique()}")

total rows: 1044
total unique students: 662


In [6]:
print(df.sample(3))

    school sex  age address famsize Pstatus  Medu  Fedu      Mjob      Fjob  \
483     GP   M   16       U     GT3       T     2     2  services     other   
523     GP   M   16       R     GT3       T     4     4   teacher   teacher   
862     MS   F   16       R     GT3       T     2     2     other  services   

     ... goout Dalc  Walc  health  absences  G1  G2  G3     subject  \
483  ...     2    1     1       3         6  12  10  11  Portuguese   
523  ...     5    2     5       4         8  14  14  15  Portuguese   
862  ...     4    1     1       2         1  14  13  14  Portuguese   

                                            student_id  
483  GP-M-16-U-GT3-T-2-2-services-other-reputation-...  
523  GP-M-16-R-GT3-T-4-4-teacher-teacher-course-yes...  
862  MS-F-16-R-GT3-T-2-2-other-services-course-yes-yes  

[3 rows x 35 columns]


### Prepareing Features for 3-Model Strategy

In [7]:
print(df_maths.shape)
print(df_maths.select_dtypes(include='number').shape)
print(df_maths.select_dtypes(include='str').shape)

(395, 34)
(395, 16)
(395, 18)


In [8]:
#  target
target = 'G3'

# feature sets
demographic_features = [col for col in df.columns if col not in ['G1', 'G2', 'G3', 'student_id']]

# Categorical columns 
cat_features = df[demographic_features].select_dtypes(include=['str']).columns.tolist()
for col in cat_features:
    df[col] = df[col].astype('category') 

# feature sets for  3 models
features_model1 = demographic_features
features_model2 = demographic_features + ['G1']
features_model3 = demographic_features + ['G1', 'G2']

In [9]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [16]:
def train_and_evaluate(df, feature_list, target_col='G3'):
    X = df[feature_list]
    y = df[target_col]
    groups = df['student_id']
    
    gkf = GroupKFold(n_splits=5)
    
    mae_scores = []
    mse_scores = []
    r2_scores = []
    
    print(f"\n--- Training Model with {len(feature_list)} features ---")
    
    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(
            tree_method="hist", 
            enable_categorical=True,
            n_estimators=1000,
            learning_rate=0.01, # Low learning rate for small data
            max_depth=4,
            early_stopping_rounds=50,
            random_state=42,
            subsample = 0.8,
            colsample_bytree = 0.8,
            reg_lambda = 0.1
            
        )
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))
        mse_scores.append(mean_squared_error(y_val, preds))
        r2_scores.append(r2_score(y_val, preds))
        
    print('MAE :',mae_scores)    
    print(f"Average MAE: {np.mean(mae_scores):.4f}")
    print('MSE :',mse_scores)    
    print(f"Average MSE: {np.mean(mse_scores):.4f}")
    print('r2_scores : ',r2_scores)
    print(f"Average R2: {np.mean(r2_scores):.4f}")
    return model

In [17]:
# Running your 3-model strategy
# Evaluating Model 1 (Pre-exam)
m1 = train_and_evaluate(df, features_model1)


--- Training Model with 31 features ---
MAE : [2.317714214324951, 2.296386241912842, 2.392878532409668, 2.2925360202789307, 2.5274226665496826]
Average MAE: 2.3654
MSE : [9.953160285949707, 9.426074981689453, 10.718029975891113, 9.995283126831055, 12.530057907104492]
Average MSE: 10.5245
r2_scores :  [0.2734776735305786, 0.24531346559524536, 0.284837007522583, 0.3787190318107605, 0.26862478256225586]
Average R2: 0.2902


In [18]:
# Evaluating Model 2 (After G1)
m2 = train_and_evaluate(df, features_model2)


--- Training Model with 32 features ---
MAE : [1.3108668327331543, 1.4026020765304565, 1.3486456871032715, 1.295121431350708, 1.3659605979919434]
Average MAE: 1.3446
MSE : [3.7038159370422363, 3.5896828174591064, 3.571425437927246, 4.049466133117676, 4.203279972076416]
Average MSE: 3.8235
r2_scores :  [0.7296431064605713, 0.7125966548919678, 0.7616958618164062, 0.7482956647872925, 0.7546559572219849]
Average R2: 0.7414


In [19]:
# Evaluating Model 3 (Final prediction)
m3 = train_and_evaluate(df, features_model3)


--- Training Model with 33 features ---
MAE : [0.8319329619407654, 0.8713709115982056, 0.8565932512283325, 0.8847137093544006, 0.9047138690948486]
Average MAE: 0.8699
MSE : [1.742478370666504, 1.8422892093658447, 1.5701247453689575, 2.5675532817840576, 2.273637533187866]
Average MSE: 1.9992
r2_scores :  [0.8728092908859253, 0.8524994850158691, 0.8952330946922302, 0.8404074907302856, 0.8672885298728943]
Average R2: 0.8656


In [49]:
df[['G3']].describe()

,G3
count,1044.000000
mean,11.341954
std,3.864796
min,0.000000
25%,10.000000
50%,11.000000
75%,14.000000
max,20.000000


In [15]:
x= np.abs(df['G3'] - df['G3'].mean())
x.mean()

np.float64(2.828742238076364)